# 05. Model Training (XGBoost, Random Forest & LightGBM)
## Objectives:
- Split raw data first and fit the preprocessor only on `X_train` (no data leakage).
- Train multi-class XGBoost, Random Forest, and LightGBM Classifiers.
- Save trained artifacts and the held-out test split into the `models/` directory.

In [1]:
import pandas as pd
import joblib

In [2]:
df = pd.read_csv('/home/patel-krish/anaconda3/BrainyBeam/SuplyChainDelay/data/syndelay_v1.csv')

In [3]:
df['profit_margin'] = df['profit_per_order'] / (df['sales'] + 1e-5)
df['discount_amount'] = df['sales'] * df['order_item_discount_rate']
df['price_to_sales_ratio'] = df['product_price'] / (df['sales_per_customer'] + 1e-5)
df['lat_lon_interaction'] = df['latitude'] * df['longitude']

In [4]:
num_cols = ['sales', 'profit_per_order', 'order_item_quantity', 'order_item_discount_rate',
            'product_price', 'sales_per_customer', 'latitude', 'longitude',
            'profit_margin', 'discount_amount', 'price_to_sales_ratio', 'lat_lon_interaction']
cat_cols = ['shipping_mode', 'market', 'customer_segment', 'department_name']

In [5]:
X = df[num_cols + cat_cols]
y = df['label']

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [9]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

In [10]:
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

joblib.dump(preprocessor, '/home/patel-krish/anaconda3/BrainyBeam/SuplyChainDelay/models/preprocessor.joblib')
joblib.dump({'X_test': X_test, 'y_test': y_test}, '/home/patel-krish/anaconda3/BrainyBeam/SuplyChainDelay/models/test_split.joblib')

['/home/patel-krish/anaconda3/BrainyBeam/SuplyChainDelay/models/test_split.joblib']

In [11]:
# XGBoost Model

In [12]:
from xgboost import XGBClassifier

In [13]:
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_proc, y_train)
joblib.dump(xgb_model, '/home/patel-krish/anaconda3/BrainyBeam/SuplyChainDelay/models/xgboost_delay_model.joblib')
print('XGBoost model trained and saved successfully!')

XGBoost model trained and saved successfully!


In [14]:
# Random Forest Model

In [15]:
from sklearn.ensemble import RandomForestClassifier

In [16]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_proc, y_train)
joblib.dump(rf_model, '/home/patel-krish/anaconda3/BrainyBeam/SuplyChainDelay/models/random_forest_delay_model.joblib')
print('Random Forest model trained and saved successfully!')

Random Forest model trained and saved successfully!


In [17]:
# LightGBM Model

In [18]:
from lightgbm import LGBMClassifier

In [19]:
lgb_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=10,
    num_leaves=63,
    min_child_samples=30,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)
lgb_model.fit(X_train_proc, y_train)
joblib.dump(lgb_model, '/home/patel-krish/anaconda3/BrainyBeam/SuplyChainDelay/models/lightgbm_delay_model.joblib')
print('LightGBM model trained and saved successfully!')

LightGBM model trained and saved successfully!
